# Spectroscopic Redshift Verification

Independent check of the deflector and source redshifts for AGEL0206 by
fitting Gaussian profiles to known absorption and emission lines in the
KCWI IFU spectrum.

Inspired by Keerthi Vasan's Keck-AGELDR2 pipeline (ESI + NIRES).

**Expected redshifts:**
- Deflector: z = 0.675 (absorption: Ca H+K, G-band, H-delta, H-gamma)
- Source: z = 1.302 (emission: [OII] 3727 → ~8580 Å observed)

**KCWI wavelength coverage:** 5625–8941 Å (observed)

## 1. Load IFU data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import sys
import importlib
sys.path.insert(0, '..')
import scripts.redshift_verify as rv
importlib.reload(rv)
from scripts.redshift_verify import (
    verify_redshift, plot_line_fits, plot_spectrum_with_lines,
    fit_cahk_doublet, ABSORPTION_LINES, EMISSION_LINES
)

plt.rcParams['figure.facecolor'] = 'white'
plt.rc('font', family='serif', size=14)
plt.rc('axes', linewidth=1.5, labelsize=16)
plt.rc('xtick', labelsize=14, direction='in')
plt.rc('ytick', labelsize=14, direction='in')

In [ ]:
# Load IFU cube
ifu_file = '../Nov17_2025_DESJ0206_RL_combined_icubes_wcs.fits'
with fits.open(ifu_file) as hdul:
    hdr = hdul[0].header
    cube = np.asarray(hdul[0].data, dtype=float)

# Build wavelength array (Angstroms, vacuum)
crval = hdr['CRVAL3']
cdelt = hdr['CD3_3']
npix = hdr['NAXIS3']
crpix = hdr.get('CRPIX3', 1.0)
pix = np.arange(npix)
lam = crval + cdelt * (pix + 1 - crpix)

# Vacuum to air conversion
def vac_to_air(lam_vac):
    s = 1e4 / lam_vac
    n = 1 + 0.0000834254 + 0.02406147 / (130 - s**2) + 0.00015998 / (38.9 - s**2)
    return lam_vac / n

lam_air = vac_to_air(lam)

# Integrated spectra
deflector_flux = np.average(cube[:, 45:64, 45:55], axis=(1, 2))
noise_flux = np.std(cube[:, 28:40, 45:70], axis=(1, 2))

print(f'Wavelength range: {lam_air[0]:.1f} - {lam_air[-1]:.1f} Angstroms (air)')
print(f'Deflector spectrum: {deflector_flux.shape}')

## 2. Full spectrum with expected line positions

In [ ]:
# Plot spectrum with deflector absorption lines at z=0.675
z_defl_guess = 0.675
fig = plot_spectrum_with_lines(lam_air, deflector_flux, z_defl_guess,
                                emission=False,
                                title='AGEL0206 deflector — absorption lines')
plt.show()

# Plot spectrum with source emission lines at z=1.302
z_src_guess = 1.302
fig = plot_spectrum_with_lines(lam_air, deflector_flux, z_src_guess,
                                emission=True,
                                title='AGEL0206 source — emission lines')
plt.show()

## 3. Deflector redshift: fit absorption lines

In [ ]:
# Fit all absorption lines visible in the KCWI range
defl_result = verify_redshift(
    lam_air, deflector_flux, z_guess=z_defl_guess,
    noise=noise_flux, emission=False, window_angstrom=30.0
)

# Print per-line results
print(f"{'Line':<20} {'z_fit':>10} {'z_err':>10} {'center (Å)':>12} {'sigma (Å)':>10}")
print('-' * 65)
for name, r in defl_result['per_line'].items():
    if r['success']:
        print(f"{name:<20} {r['z_fit']:10.5f} {r['z_err']:10.5f} "
              f"{r.get('center', np.nan):12.2f} {r.get('sigma', np.nan):10.2f}")
    else:
        print(f"{name:<20} {'FAILED':>10} {r.get('message', ''):>30}")

print(f"\nWeighted average z = {defl_result['z_weighted']:.5f} ± {defl_result['z_weighted_err']:.5f}")
print(f"Median z = {defl_result['z_median']:.5f}")
print(f"Lines fit: {defl_result['n_lines_fit']}")
print(f"\nInput z = 0.67511 (from ppxf)")
print(f"Delta z = {defl_result['z_weighted'] - 0.67511:.5f}")

In [ ]:
# Plot individual line fits
fig = plot_line_fits(defl_result, title='Deflector absorption line fits')
plt.show()

## 4. Ca H+K doublet fit (tied velocity width)

In [ ]:
# Fit Ca H+K doublet with shared width
cahk = fit_cahk_doublet(lam_air, deflector_flux, z_guess=z_defl_guess,
                         window_angstrom=60.0, noise=noise_flux)

if cahk['success']:
    print(f"Ca H+K doublet fit:")
    print(f"  z(Ca K) = {cahk['z_cak']:.5f} ± {cahk['z_cak_err']:.5f}")
    print(f"  z(Ca H) = {cahk['z_cah']:.5f} ± {cahk['z_cah_err']:.5f}")
    print(f"  z(avg)  = {cahk['z_fit']:.5f} ± {cahk['z_err']:.5f}")
    print(f"  sigma   = {cahk['sigma']:.2f} Å")

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.step(cahk['wl_window'], cahk['flux_window'], 'k', lw=1, label='Data')
    ax.plot(cahk['wl_window'], cahk['model'], 'r', lw=1.5, label='Fit')
    ax.set_xlabel('Wavelength (Å)')
    ax.set_ylabel('Flux')
    ax.set_title(f"Ca H+K doublet — z = {cahk['z_fit']:.5f} ± {cahk['z_err']:.5f}")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.show()
else:
    print(f"Ca H+K fit failed: {cahk.get('message', 'unknown')}")

## 5. Source redshift: look for [OII] emission

In [ ]:
# Source [OII] 3727 at z=1.302 would appear at ~8580 Å
# This is near the red edge of KCWI coverage (~8941 Å)
# Also check for [OII] in the lensed arc region (different spaxels)

# Try the integrated deflector spectrum first (may contain source emission)
src_result = verify_redshift(
    lam_air, deflector_flux, z_guess=z_src_guess,
    noise=noise_flux, emission=True, window_angstrom=30.0
)

print(f"Source emission line search (z_guess={z_src_guess}):")
print(f"{'Line':<20} {'z_fit':>10} {'z_err':>10}")
print('-' * 45)
for name, r in src_result['per_line'].items():
    if r['success']:
        print(f"{name:<20} {r['z_fit']:10.5f} {r['z_err']:10.5f}")
    else:
        print(f"{name:<20} {'FAILED':>10}")

if src_result['n_lines_fit'] > 0:
    print(f"\nWeighted z = {src_result['z_weighted']:.5f} ± {src_result['z_weighted_err']:.5f}")
    fig = plot_line_fits(src_result, title='Source emission line fits')
    plt.show()
else:
    print("\nNo source emission lines detected in the integrated spectrum.")
    print("Try extracting from arc-dominated spaxels instead.")

## 6. Source emission from arc spaxels

If the source [OII] emission is diluted in the integrated spectrum,
extract from spaxels dominated by the lensed arc.

In [ ]:
# White-light image to identify arc spaxels
fig, ax = plt.subplots(figsize=(8, 8))
whitelight = np.sum(cube[:, 35:75, 30:70], axis=0)
im = ax.imshow(whitelight, origin='lower', cmap='viridis')
plt.colorbar(im, ax=ax, label='Summed flux')
ax.set_title('White-light image — identify arc spaxels')
ax.set_xlabel('X spaxel')
ax.set_ylabel('Y spaxel')

# Mark the deflector region for reference
from matplotlib.patches import Rectangle
defl_rect = Rectangle((45-30, 45-35), 10, 19, linewidth=2,
                       edgecolor='red', facecolor='none', label='Deflector')
ax.add_patch(defl_rect)
ax.legend()
plt.show()

print("Adjust the arc spaxel region below based on the white-light image.")
print("Look for extended emission offset from the deflector center.")

In [ ]:
# Extract arc spectrum from candidate arc spaxels
# ADJUST these indices based on the white-light image above
# These are indices into the FULL cube (not the 35:75, 30:70 subregion)
arc_y = slice(50, 65)  # adjust
arc_x = slice(56, 62)  # adjust — offset from deflector center

arc_flux = np.average(cube[:, arc_y, arc_x], axis=(1, 2))

# Subtract deflector continuum (rough)
arc_minus_defl = arc_flux - deflector_flux

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
axes[0].step(lam_air, arc_flux, 'k', lw=0.8, label='Arc spaxels')
axes[0].step(lam_air, deflector_flux, 'r', lw=0.8, alpha=0.5, label='Deflector')
axes[0].set_ylabel('Flux')
axes[0].set_title('Arc vs deflector spectra')
axes[0].legend()

axes[1].step(lam_air, arc_minus_defl, 'b', lw=0.8, label='Arc - Deflector')
# Mark expected [OII] position
oii_obs = 3727.4 * (1 + z_src_guess)
axes[1].axvline(oii_obs, color='green', ls='--', alpha=0.7, label=f'[OII] at z={z_src_guess}')
axes[1].set_xlabel('Wavelength (Å)')
axes[1].set_ylabel('Flux')
axes[1].set_title('Residual (arc - deflector) — look for source emission')
axes[1].legend()

plt.tight_layout()
plt.show()

# Try fitting source emission in the residual
src_arc_result = verify_redshift(
    lam_air, arc_minus_defl, z_guess=z_src_guess,
    noise=noise_flux, emission=True, window_angstrom=40.0
)

print(f"\nSource emission in arc residual:")
for name, r in src_arc_result['per_line'].items():
    status = f"z={r['z_fit']:.5f} ± {r['z_err']:.5f}" if r['success'] else 'FAILED'
    print(f"  {name}: {status}")
if src_arc_result['n_lines_fit'] > 0:
    print(f"  Weighted z = {src_arc_result['z_weighted']:.5f} ± {src_arc_result['z_weighted_err']:.5f}")

## 7. Summary

In [ ]:
print('=' * 60)
print('REDSHIFT VERIFICATION SUMMARY — AGEL0206')
print('=' * 60)
print(f"\nDeflector (absorption lines):")
print(f"  Weighted z = {defl_result['z_weighted']:.5f} ± {defl_result['z_weighted_err']:.5f}")
print(f"  Median z   = {defl_result['z_median']:.5f}")
print(f"  N lines    = {defl_result['n_lines_fit']}")
print(f"  ppxf z     = 0.67511 (from streamlined notebook)")
print(f"  Delta z    = {defl_result['z_weighted'] - 0.67511:+.5f}")
if cahk['success']:
    print(f"  Ca H+K z   = {cahk['z_fit']:.5f} ± {cahk['z_err']:.5f}")

print(f"\nSource (emission lines):")
if src_result['n_lines_fit'] > 0:
    print(f"  Weighted z = {src_result['z_weighted']:.5f} ± {src_result['z_weighted_err']:.5f}")
elif src_arc_result['n_lines_fit'] > 0:
    print(f"  Arc residual z = {src_arc_result['z_weighted']:.5f} ± {src_arc_result['z_weighted_err']:.5f}")
else:
    print(f"  No source emission lines detected")
    print(f"  Expected z = 1.302 ([OII] → {3727.4*(1+1.302):.0f} Å)")
print('=' * 60)